<a href="https://colab.research.google.com/github/luqthewolf-lgtm/Guia-Defijitivo/blob/main/parte3_definitivo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Guia Definitivo -- Parte 3 de 5
## SVM + Redes Neurais + Clustering + PCA + MLflow

**Pre-requisito:** Partes 1 e 2 concluidas

**Modulos desta parte:**
- Modulo 9: SVM -- **Prioridade: ALTA**
- Modulo 10: Redes Neurais -- **Prioridade: MEDIA**
- Modulo 11: Clustering e PCA -- **Prioridade: ALTA**
- Modulo 12: MLflow -- **Prioridade: MEDIA (entrevista e producao)**

---

## Como usar este notebook

Cada linha de codigo tem comentario explicando o que faz e por que.
Antes de executar qualquer celula, leia o bloco de markdown acima dela.
Nas questoes: resolva no papel, so entao veja o gabarito.

---

## Glossario de termos desta parte

| Termo | Definicao |
|-------|-----------|
| Hiperplano | Generalizacao de reta/plano para n dimensoes -- fronteira do SVM |
| Margem | Distancia entre o hiperplano e os pontos mais proximos de cada classe |
| Support Vectors | Pontos que ficam na margem -- definem o hiperplano |
| Kernel | Funcao que mede similaridade entre pontos em espaco de alta dimensao |
| Kernel RBF | Radial Basis Function -- kernel gaussiano, mais comum no SVM |
| Gamma | Raio de influencia de cada ponto no kernel RBF |
| Platt Scaling | Metodo para converter outputs do SVM em probabilidades calibradas |
| Neuronio | Unidade basica de uma rede neural: recebe inputs, aplica ativacao |
| Ativacao | Funcao nao-linear aplicada apos combinacao linear no neuronio |
| ReLU | max(0,x) -- nunca negativo, funcao de ativacao mais comum |
| Sigmoid | 1/(1+e^-x) -- saida entre 0 e 1, nunca negativa |
| Tanh | tanh(x) -- saida entre -1 e 1, pode ser negativa |
| Backpropagation | Algoritmo para calcular gradientes em redes neurais |
| Clustering | Agrupar dados em clusters sem usar rotulos (nao-supervisionado) |
| Centroide | Ponto central de um cluster no K-means |
| Inertia | Soma das distancias de cada ponto ao seu centroide (K-means) |
| Silhouette | Metrica de qualidade dos clusters: de -1 a 1, maior = melhor |
| Dendrograma | Arvore hierarquica que mostra como clusters sao fundidos |
| Linkage | Criterio de distancia entre clusters no hierarchical clustering |
| PCA | Principal Component Analysis: reducao de dimensionalidade |
| Componente Principal | Direcao de maxima variancia dos dados |
| Variancia Explicada | Proporcao da variancia total capturada por cada componente |
| Loading | Contribuicao de cada variavel original para um componente |
| MLflow | Plataforma para rastrear experimentos e versionar modelos de ML |
| Run | Uma execucao de experimento no MLflow |
| Experiment | Grupo de runs no MLflow (ex: 'modelo_propensao_pj') |
| Artifact | Arquivo salvo no MLflow (modelo, graficos, dados) |
| Drift | Mudanca na distribuicao dos dados em producao vs treino |
| PSI | Population Stability Index: mede o quanto a distribuicao mudou |


---
# Setup -- Execute antes de qualquer celula

In [ ]:
# Instalando dependencias
!pip install numpy pandas scikit-learn matplotlib seaborn scipy xgboost mlflow --quiet
print("Instalacao concluida!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.5/838.5 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207

In [ ]:
# ============================================================
# IMPORTS -- cada biblioteca com sua funcao
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
import warnings, math, os
warnings.filterwarnings('ignore')

# Modelos
from sklearn.svm import SVC, SVR            # Support Vector Machine
from sklearn.neural_network import MLPClassifier  # Multi-layer Perceptron
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

# Clustering e reducao de dimensionalidade
from sklearn.cluster import KMeans          # K-means clustering
from sklearn.decomposition import PCA       # Principal Component Analysis
from sklearn.preprocessing import StandardScaler

# Pipeline e avaliacao
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    train_test_split, KFold, StratifiedKFold,
    cross_validate, GridSearchCV
)
from sklearn.metrics import (
    roc_auc_score, log_loss, mean_squared_error,
    confusion_matrix, classification_report,
    silhouette_score
)
from sklearn.datasets import make_classification, make_regression

# XGBoost e MLflow
from xgboost import XGBClassifier
import mlflow
import mlflow.sklearn

np.random.seed(42)
plt.rcParams.update({
    'figure.figsize': (12, 5),
    'axes.grid': True,
    'grid.alpha': 0.3
})
print("Todos os imports carregados com sucesso!")

Todos os imports carregados com sucesso!


---
# MODULO 9 -- Support Vector Machines (SVM)
## Prioridade: ALTA | ISLP Capitulo 9

---

## A analogia do separador de documentos

Imagine uma mesa com documentos de dois tipos misturados:
contratos aprovados (azul) e contratos reprovados (vermelho).
Voce precisa tracar uma linha que separe os dois grupos.

Ha infinitas linhas possiveis que separam corretamente.
O SVM escolhe a linha que fica o mais longe possivel dos documentos
mais proximos de cada grupo -- maximiza a MARGEM de seguranca.

Os documentos que ficam exatamente na borda da margem sao os
**support vectors** -- sao eles que definem onde a linha passa.
Se voce remover qualquer outro documento que nao seja support vector,
a linha nao muda. Isso e a elegancia matematica do SVM.

---

## 9A. O hiperplano de maxima margem

Em 2D: linha. Em 3D: plano. Em n dimensoes: hiperplano.

O SVM resolve:
```
maximize: 2/||w||        (maximizar a margem)
sujeito a: y_i(w*x_i + b) >= 1 para todo i
```

Onde w e o vetor normal ao hiperplano e b e o intercepto.

**Margem suave (Soft Margin):**
Dados reais raramente sao linearmente separaveis.
O SVM permite violacoes controladas pelo parametro C:
- C pequeno: aceita mais violacoes, margem mais larga, modelo mais simples
- C grande: aceita poucas violacoes, margem mais estreita, modelo mais complexo

---

## 9B. O kernel trick -- magia matematica

E se os dados nao forem linearmente separaveis em nenhuma dimensao?

O kernel trick mapeia os dados para um espaco de maior dimensao
onde eles SE TORNAM separaveis, sem calcular explicitamente esse mapeamento.

**Kernel RBF (Radial Basis Function):**
```
K(x_i, x_j) = exp(-gamma * ||x_i - x_j||^2)
```
Mede similaridade como funcao da distancia.
Gamma controla o raio de influencia:
- Gamma pequeno: cada ponto influencia uma regiao GRANDE (modelo suave)
- Gamma grande: cada ponto influencia uma regiao PEQUENA (modelo complexo)

---

## 9C. Parametros e seus efeitos -- o que a prova cobra

| Parametro | Efeito pequeno | Efeito grande |
|-----------|----------------|---------------|
| **C** | Margem larga, simples, underfitting | Margem estreita, complexo, overfitting |
| **Gamma** (RBF) | Raio grande, suave, underfitting | Raio pequeno, complexo, overfitting |

**REGRAS QUE NAO ERRAM:**
1. SEMPRE normalizar antes do SVM (usa distancias euclidanas -- escala importa)
2. C pequeno = regularizacao FORTE (C = 1/lambda, como na logistica)
3. Gamma alto = overfitting no kernel RBF
4. SVM nao produz probabilidades nativas -- usa Platt Scaling com probability=True
5. Logistica: probabilidade calibrada por construcao. SVM: aproximada via Platt.

---

## 9D. SVM vs Logistica -- quando usar cada um

| Situacao | Preferir |
|----------|----------|
| n >> p (muito dado, poucas features) | Logistica |
| p >> n (muitas features, pouco dado) | SVM com kernel linear |
| Fronteira de decisao complexa nao-linear | SVM com kernel RBF |
| Precisa de probabilidades bem calibradas | Logistica |
| Precisa de velocidade de treinamento | Logistica |
| Dataset muito grande (> 100k) | Logistica ou XGBoost (SVM escala mal) |


In [ ]:
# ============================================================
# 9.1 -- SVM: NORMALIZACAO, C E GAMMA
# ============================================================

np.random.seed(42)

# Dataset de classificacao para testar o SVM
# Simula base de clientes PJ para deteccao de detratores
X9, y9 = make_classification(
    n_samples=700,
    n_features=10,
    n_informative=7,
    random_state=42
)

kf9 = KFold(n_splits=5, shuffle=False)   # padrao da prova Itau

print("=== 1. NORMALIZACAO E OBRIGATORIA NO SVM ===")
print()
print("SVM usa distancias euclidianas entre pontos.")
print("Variaveis em escala maior dominam o calculo de distancia injustamente.")
print()

# SVM sem normalizacao
res_sem = cross_validate(
    SVC(kernel='rbf', C=1.0, probability=True),  # SVM sem preprocessamento
    X9, y9,
    cv=kf9,
    scoring='roc_auc'
)

# SVM com normalizacao dentro de um Pipeline (correto)
pipe_com = Pipeline([
    ('scaler', StandardScaler()),                  # passo 1: normalizar (z-score)
    ('svm', SVC(kernel='rbf', C=1.0, probability=True))  # passo 2: SVM
])
res_com = cross_validate(pipe_com, X9, y9, cv=kf9, scoring='roc_auc')

print(f"  AUC sem normalizar: {res_sem['test_score'].mean():.4f}  <- degradado!")
print(f"  AUC com normalizar: {res_com['test_score'].mean():.4f}  <- correto")
print()
print("[!] Q29 da prova: 'SVM nao precisa normalizar' -> FALSO")
print()

print("=== 2. EFEITO DO PARAMETRO C ===")
print()
print("C controla o tradeoff entre margem larga e classificacao correta no treino.")
print("C pequeno = regularizacao forte = margem larga = modelo simples")
print("C grande  = regularizacao fraca = margem estreita = modelo complexo")
print()
print(f"{'C':>8} | {'Treino':>8} | {'Val':>8} | {'Gap':>7} | Diagnostico")
print("-" * 60)

for C_val in [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]:
    pipe_c = Pipeline([
        ('scaler', StandardScaler()),
        ('svm', SVC(kernel='rbf', C=C_val, probability=True))
    ])
    res_c = cross_validate(pipe_c, X9, y9, cv=kf9,
                            scoring='roc_auc', return_train_score=True)
    tr_c = res_c['train_score'].mean()
    va_c = res_c['test_score'].mean()

    if C_val < 0.01:   diag = "underfitting: margem larga demais"
    elif C_val > 10.0: diag = "overfitting: margem estreita demais"
    else:              diag = "OK"
    print(f"{C_val:>8.3f} | {tr_c:>8.4f} | {va_c:>8.4f} | {tr_c-va_c:>7.4f} | {diag}")

print()
print("=== 3. EFEITO DO GAMMA (kernel RBF) ===")
print()
print("Gamma controla o raio de influencia de cada ponto.")
print("Gamma pequeno: cada ponto influencia uma regiao grande -> modelo suave")
print("Gamma grande:  cada ponto influencia uma regiao pequena -> modelo complexo")
print()
print(f"{'Gamma':>10} | {'Treino':>8} | {'Val':>8} | Diagnostico")
print("-" * 55)

for gamma_val in [0.001, 0.01, 0.1, 1.0, 10.0]:
    pipe_g = Pipeline([
        ('scaler', StandardScaler()),
        ('svm', SVC(kernel='rbf', C=1.0, gamma=gamma_val, probability=True))
    ])
    res_g = cross_validate(pipe_g, X9, y9, cv=kf9,
                            scoring='roc_auc', return_train_score=True)
    tr_g = res_g['train_score'].mean()
    va_g = res_g['test_score'].mean()

    if gamma_val < 0.01:   diag = "raio grande -> underfitting"
    elif gamma_val >= 1.0: diag = "raio pequeno -> overfitting"
    else:                  diag = "OK"
    print(f"{gamma_val:>10.3f} | {tr_g:>8.4f} | {va_g:>8.4f} | {diag}")

print()
print("[!] Q16 da prova: Gamma alto + kernel RBF -> overfitting")
print("    Cada ponto influencia regiao minuscula -> memoriza o treino")

=== 1. NORMALIZACAO E OBRIGATORIA NO SVM ===

SVM usa distancias euclidianas entre pontos.
Variaveis em escala maior dominam o calculo de distancia injustamente.

  AUC sem normalizar: 0.9282  <- degradado!
  AUC com normalizar: 0.9357  <- correto

[!] Q29 da prova: 'SVM nao precisa normalizar' -> FALSO

=== 2. EFEITO DO PARAMETRO C ===

C controla o tradeoff entre margem larga e classificacao correta no treino.
C pequeno = regularizacao forte = margem larga = modelo simples
C grande  = regularizacao fraca = margem estreita = modelo complexo

       C |   Treino |      Val |     Gap | Diagnostico
------------------------------------------------------------
   0.001 |   0.8859 |   0.8683 |  0.0176 | underfitting: margem larga demais
   0.010 |   0.8859 |   0.8681 |  0.0177 | OK
   0.100 |   0.9207 |   0.8935 |  0.0271 | OK
   1.000 |   0.9718 |   0.9357 |  0.0361 | OK
  10.000 |   0.9937 |   0.9549 |  0.0388 | OK
 100.000 |   1.0000 |   0.9382 |  0.0618 | overfitting: margem estreita de

In [ ]:
# ============================================================
# 9.2 -- SVM: PROBABILIDADE E PLATT SCALING
# ============================================================

print("=== SVM E PROBABILIDADE: Platt Scaling ===")
print()
print("O SVM nao foi projetado para produzir probabilidades.")
print("Ele produz uma pontuacao de decisao (distancia ao hiperplano).")
print("Para converter em probabilidade, usa Platt Scaling:")
print("  P(y=1|x) = sigmoid(A * score + B)")
print("  onde A e B sao calibrados em um conjunto de validacao interno.")
print()
print("Resultado: probabilidades aproximadas, menos calibradas que logistica.")
print()

np.random.seed(42)
X9b, y9b = make_classification(n_samples=500, n_features=8, random_state=42)
Xtr9b, Xte9b, ytr9b, yte9b = train_test_split(X9b, y9b, test_size=0.2, random_state=42)

# Comparando logistica vs SVM na calibracao das probabilidades
pipe_log = Pipeline([('sc', StandardScaler()), ('lr', LogisticRegression())])
pipe_svm = Pipeline([('sc', StandardScaler()), ('svm', SVC(probability=True))])

pipe_log.fit(Xtr9b, ytr9b)
pipe_svm.fit(Xtr9b, ytr9b)

proba_log = pipe_log.predict_proba(Xte9b)[:,1]
proba_svm = pipe_svm.predict_proba(Xte9b)[:,1]

auc_log = roc_auc_score(yte9b, proba_log)
auc_svm = roc_auc_score(yte9b, proba_svm)
ll_log  = log_loss(yte9b, proba_log)
ll_svm  = log_loss(yte9b, proba_svm)

print(f"{'Modelo':<20} {'AUC':>8} {'Log Loss':>10}  Notas")
print("-" * 60)
print(f"{'Logistica':<20} {auc_log:>8.4f} {ll_log:>10.4f}  Calibrada por construcao")
print(f"{'SVM (Platt)':<20} {auc_svm:>8.4f} {ll_svm:>10.4f}  Aproximada via Platt Scaling")
print()

# Visualizando a distribuicao das probabilidades previstas
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, proba, nome, cor in zip(
    axes,
    [proba_log, proba_svm],
    ['Logistica', 'SVM (Platt Scaling)'],
    ['steelblue', 'coral']
):
    # Histograma separado por classe real
    ax.hist(proba[yte9b==0], bins=20, alpha=0.6, color='steelblue',
            label='Real: classe 0', density=True)
    ax.hist(proba[yte9b==1], bins=20, alpha=0.6, color='coral',
            label='Real: classe 1', density=True)
    ax.axvline(0.5, color='black', linestyle='--', lw=1.5, label='Threshold=0.5')
    ax.set_xlabel('Probabilidade prevista')
    ax.set_title(f'{nome}
Boa separacao = classes bem separadas nas probabilidades')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print("INTERPRETACAO:")
print("  Logistica: probabilidades bem calibradas por construcao.")
print("  Se o modelo diz P=0.7, em media 70% desses casos sao positivos.")
print()
print("  SVM: probabilidades aproximadas via Platt Scaling.")
print("  Menos calibradas, mas AUC pode ser similar ou melhor em alguns casos.")
print()
print("[!] Q27: 'Arvore de decisao tem probabilidades calibradas' -> FALSO")
print("  Arvores e Random Forest tambem NAO sao calibrados.")
print("  Para calibrar: usar CalibratedClassifierCV do sklearn.")
print("  Logistica: unico modelo classico calibrado por construcao.")

SyntaxError: unterminated f-string literal (detected at line 57) (2716736572.py, line 57)

In [ ]:
# ============================================================
# 9.3 -- SVR: Reproducindo Q10 e Q11 da prova
# ============================================================

from sklearn.linear_model import ElasticNet

print("=== SVR: REPRODUCAO DAS QUESTOES DA PROVA ===")
print()
print("Padrao CONFIRMADO com dados reais: KFold(n_splits=K, shuffle=False)")
print()

kf5_prova  = KFold(n_splits=5,  shuffle=False)
kf10_prova = KFold(n_splits=10, shuffle=False)

# Q10 -- Elastic Net
if os.path.exists('regressao_Q1.csv'):
    df_q10 = pd.read_csv('regressao_Q1.csv')
    X_q10 = df_q10.drop('target', axis=1)
    y_q10 = df_q10['target']

    m_q10 = ElasticNet(alpha=1.0, l1_ratio=0.01)
    r_q10 = cross_validate(m_q10, X_q10, y_q10, cv=kf5_prova,
                            scoring='neg_mean_squared_error',
                            return_train_score=True)
    print(f"Q10 -- Elastic Net (alpha=1.0, l1_ratio=0.01, KFold5 sem shuffle):")
    print(f"  Treino MSE: {-r_q10['train_score'].mean():.4f}")
    print(f"  Val MSE:    {-r_q10['test_score'].mean():.4f}")
    print(f"  Gabarito:   ~0.2686 e ~0.2693")
else:
    print("Q10: coloque regressao_Q1.csv na mesma pasta do notebook.")

print()

# Q11 -- SVR linear
if os.path.exists('regressao_Q2.csv'):
    df_q11 = pd.read_csv('regressao_Q2.csv')
    X_q11 = df_q11.drop('target', axis=1)
    y_q11 = df_q11['target']

    m_q11 = SVR(kernel='linear', C=0.001)
    r_q11 = cross_validate(m_q11, X_q11, y_q11, cv=kf5_prova,
                            scoring='neg_mean_squared_error',
                            return_train_score=True)
    print(f"Q11 -- SVR (kernel='linear', C=0.001, KFold5 sem shuffle):")
    print(f"  Treino MSE: {-r_q11['train_score'].mean():.0f}")
    print(f"  Val MSE:    {-r_q11['test_score'].mean():.0f}")
    print(f"  Gabarito:   ~20199 e ~20207")
else:
    print("Q11: coloque regressao_Q2.csv na mesma pasta do notebook.")

print()
print("[!] C=0.001 em SVR = regularizacao MUITO FORTE")
print("    C = 1/lambda, entao C pequeno = penalidade grande.")
print("    Por isso o SVR com C=0.001 tem desempenho degradado (underfitting).")

In [ ]:
# ============================================================
# QUESTOES DO MODULO 9
# ============================================================

print("=" * 65)
print("QUESTOES DO MODULO 9 -- Resolva antes de ver")
print("=" * 65)
print()
print("QUESTAO 1 (regras basicas):")
print("  Assinale VERDADEIRO ou FALSO para cada afirmacao sobre SVM:")
print("  (a) SVM sempre precisa de normalizacao dos dados")
print("  (b) C=0.001 em SVM significa regularizacao fraca")
print("  (c) Gamma alto no kernel RBF tende a causar underfitting")
print("  (d) SVM com probability=True produz probabilidades exatas")
print("  (e) Aumentar C aumenta a margem de separacao")
print()
print("QUESTAO 2 (diagnostico):")
print("  SVM com kernel RBF, C=1.0, Gamma=10.0.")
print("  AUC treino = 0.99, AUC validacao = 0.61.")
print("  O que esta acontecendo e como corrigir?")
print()
print("QUESTAO 3 (escolha do modelo):")
print("  Voce tem um dataset com 500 clientes e 200 features (p >> n).")
print("  Qual kernel do SVM seria mais adequado? Por que?")
print()
print("-" * 65)
print()
print("GABARITOS:")
print()
print("Q1:")
print("  (a) VERDADEIRO. SVM usa distancias euclidianas -- escala importa.")
print("      Sem normalizar, variaveis em maior escala dominam injustamente.")
print()
print("  (b) FALSO. C=0.001 significa regularizacao FORTE.")
print("      C = 1/lambda. C pequeno = lambda grande = penalidade alta.")
print("      C=0.001 -> lambda=1000 -> regularizacao muito forte -> underfitting.")
print()
print("  (c) FALSO. Gamma alto tende a causar OVERFITTING.")
print("      Gamma grande = raio pequeno = cada ponto influencia regiao minuscula.")
print("      O modelo memoriza cada ponto do treino -> overfitting.")
print()
print("  (d) FALSO. probability=True usa Platt Scaling, que produz")
print("      probabilidades APROXIMADAS, nao exatas.")
print("      Logistica e o unico modelo classico calibrado por construcao.")
print()
print("  (e) FALSO. Aumentar C DIMINUI a margem (aceita menos violacoes).")
print("      C pequeno -> margem larga -> modelo simples.")
print("      C grande  -> margem estreita -> modelo complexo.")
print()
print("Q2: OVERFITTING severo. Gamma=10.0 e muito alto.")
print("  Cada ponto influencia uma regiao minuscula.")
print("  O modelo memoriza cada ponto do treino -> AUC treino perfeito.")
print("  Em dados novos: falha completamente.")
print()
print("  COMO CORRIGIR:")
print("  1. Reduzir Gamma: testar 0.001, 0.01, 0.1 via GridSearchCV")
print("  2. Aumentar C ligeiramente se o modelo underfittar")
print("  3. Usar Pipeline com StandardScaler")
print("  4. GridSearchCV para encontrar C e Gamma otimos juntos")
print()
print("Q3: Kernel linear.")
print("  Com p >> n (mais features que amostras), o espaco de alta dimensao")
print("  JA permite separacao linear (Maldimensionalidade -- Cover's theorem).")
print("  Kernel RBF com p grande tende a overfittar (muita liberdade).")
print("  Kernel linear e mais eficiente e generaliza melhor neste cenario.")
print("  Alternativa: Lasso (selecao de features) + logistica.")

---
# MODULO 10 -- Redes Neurais e Deep Learning
## Prioridade: MEDIA | ISLP Capitulo 10

---

## A analogia do cerebro artificial

O cerebro tem bilhoes de neuronios conectados.
Cada neuronio recebe sinais de outros, combina-os e dispara (ou nao).
Aprender e ajustar a forca das conexoes (sinapses) com base na experiencia.

Uma rede neural artificial simula isso:
- Cada **neuronio** recebe inputs, calcula uma combinacao linear e aplica uma funcao nao-linear
- As conexoes tem **pesos** (como sinapses) que sao ajustados durante o treino
- Aprender = encontrar os pesos que minimizam o erro nas previsoes

---

## 10A. Estrutura de uma rede

```
Input Layer  ->  [Hidden Layers]  ->  Output Layer
   X1                                    P(Y=1)
   X2   ->   w1*X1 + w2*X2 + b   ->   sigmoid
   X3         ->  ativacao  ->
```

Cada neuronio faz:
1. Combinacao linear: z = w1*x1 + w2*x2 + ... + b
2. Funcao de ativacao: a = f(z)

Sem a funcao de ativacao, a rede inteira colapsa para uma transformacao linear.

---

## 10B. Funcoes de ativacao -- o que a prova cobra

| Funcao | Formula | Range | Negativo? | Uso tipico |
|--------|---------|-------|-----------|-----------|
| **ReLU** | max(0, x) | [0, +inf) | **NUNCA** | Camadas ocultas (padrao) |
| **Leaky ReLU** | max(0.01x, x) | (-inf, +inf) | **PODE** | Evita dying ReLU |
| **Sigmoid** | 1/(1+e^-x) | (0, 1) | **NUNCA** | Saida binaria |
| **Tanh** | tanh(x) | (-1, 1) | **PODE** | Camadas ocultas (centralizada) |
| **Linear** | x | (-inf, +inf) | Pode | Saida de regressao |
| **Softmax** | e^xi / sum(e^xj) | (0,1), soma=1 | Nunca | Saida multiclasse |

**[!] Q17 da prova:** qual NAO pode gerar saida de -0.001?
- ReLU: max(0,x) -> output >= 0 sempre -> **IMPOSSIVEL gerar negativo**
- Sigmoid: entre 0 e 1 -> **IMPOSSIVEL gerar negativo**
- Leaky ReLU: para x<0, f(x)=0.01x -> PODE gerar negativo
- Tanh: entre -1 e 1 -> PODE gerar negativo

**[!] Q28 da prova:** rede com ativacoes LINEARES em todas as camadas:
Linear(Linear(Linear(x))) = Linear(x)
Profundidade nao adiciona nada! A rede inteira equivale a uma unica camada linear.
Por isso funcoes nao-lineares sao ESSENCIAIS nas camadas ocultas.

---

## 10C. Quando usar deep learning

**USE quando:**
- Dados de imagem, texto, audio (dados nao-estruturados)
- Dataset muito grande (> 1M amostras)
- Relacoes extremamente complexas

**NAO USE quando (contexto bancario tipico):**
- Dados tabulares estruturados -> XGBoost supera com muito menos dado
- Dataset pequeno/medio (< 100k) -> RF ou XGBoost mais estavel
- Necessidade de interpretabilidade -> logistica ou arvore
- Restricao regulatoria (BACEN) -> modelo explicavel obrigatorio

---

## 10D. Backpropagation (intuicao)

Como a rede aprende? Backpropagation calcula o gradiente
do erro em relacao a cada peso, usando a regra da cadeia.

```
erro -> d_erro/d_pesos_ultima_camada -> d_erro/d_pesos_penultima -> ...
```

O otimizador (SGD, Adam) usa o gradiente para atualizar os pesos:
```
w_novo = w_antigo - learning_rate * gradiente
```

Learning rate muito alto: pulos grandes, pode nao convergir.
Learning rate muito baixo: converge devagar demais.


In [ ]:
# ============================================================
# 10.1 -- FUNCOES DE ATIVACAO: VISUALIZACAO COMPLETA
# ============================================================

x_ativ = np.linspace(-5, 5, 400)   # 400 pontos de -5 a +5

# Definindo cada funcao de ativacao
funcoes = {
    'ReLU  max(0,x)\nNUNCA negativo': lambda x: np.maximum(0, x),
    'Leaky ReLU  max(0.01x, x)\nPODE ser negativo': lambda x: np.where(x > 0, x, 0.01*x),
    'Sigmoid  1/(1+e^-x)\n(0,1) NUNCA negativo': lambda x: 1/(1+np.exp(-x)),
    'Tanh  tanh(x)\n(-1,1) PODE ser negativo': lambda x: np.tanh(x),
    'Linear  f(x)=x\nRede colapsa se so usada': lambda x: x,
    'Softmax (2 classes)\n(0,1) soma=1, nunca neg.': lambda x: np.exp(x)/(np.exp(x)+np.exp(-x)),
}

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for ax, (nome, func) in zip(axes.ravel(), funcoes.items()):
    y_f = func(x_ativ)
    ax.plot(x_ativ, y_f, lw=3, color='steelblue')
    ax.axhline(0, color='black', lw=0.8, linestyle='--')  # linha do zero
    ax.axvline(0, color='black', lw=0.8, linestyle='--')  # linha vertical
    ax.set_title(nome, fontsize=10)
    ax.set_ylim(-2, 2)
    ax.set_xlim(-5, 5)

plt.suptitle('Funcoes de Ativacao -- Guia Completo para a Prova', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print("=== TESTE RAPIDO: qual pode gerar -0.001? ===")
print()

x_test = -0.1   # valor negativo para testar

funcoes_simples = [
    ('ReLU',       lambda x: max(0, x)),
    ('Leaky ReLU', lambda x: x if x > 0 else 0.01*x),
    ('Sigmoid',    lambda x: 1/(1+np.exp(-x))),
    ('Tanh',       lambda x: np.tanh(x)),
    ('Linear',     lambda x: x),
]

print(f"Para x = {x_test}:")
print()
for nome_f, func_f in funcoes_simples:
    output = func_f(x_test)
    pode   = "PODE ser negativo" if output < 0 else "NUNCA negativo (output >= 0)"
    print(f"  {nome_f:<15}: f({x_test}) = {output:.4f}  -> {pode}")

print()
print("[!] Q17 da prova: qual NAO pode gerar -0.001?")
print("    ReLU: max(0, x) -> sempre >= 0 -> IMPOSSIVEL gerar negativo")
print("    Sigmoid: entre 0 e 1 -> IMPOSSIVEL gerar negativo")
print("    Leaky ReLU: 0.01*x para x<0 -> PODE gerar negativo")
print("    Tanh: entre -1 e 1 -> PODE gerar negativo")
print()
print("[!] Q28 da prova: rede com ativacoes lineares em todas as camadas")
print("    f(f(f(x))) onde f(x)=x -> ainda e f(x)=x (transformacao linear!)")
print("    Nao importa quantas camadas: resultado identico a 1 camada linear.")
print("    Profundidade so adiciona poder com funcoes NAO-lineares.")

In [ ]:
# ============================================================
# 10.2 -- MLP PARA DADOS TABULARES: QUANDO VALE A PENA?
# ============================================================

np.random.seed(42)
X10, y10 = make_classification(
    n_samples=1000,
    n_features=15,
    n_informative=8,
    random_state=42
)
X_tr10, X_te10, y_tr10, y_te10 = train_test_split(X10, y10, test_size=0.2, random_state=42)

kf10 = KFold(5, shuffle=False)

print("=== COMPARACAO: MLP vs XGBoost em dados tabulares ===")
print()
print("Contexto: dataset tipico de propensao (1000 clientes, 15 features)")
print()
print(f"{'Modelo':<35} {'AUC Val':>9} {'Std':>7} | Observacao")
print("-" * 70)

modelos10 = [
    ("Logistica (baseline)",
     Pipeline([('sc', StandardScaler()), ('lr', LogisticRegression(max_iter=1000))])),

    ("MLP (1 camada, 50 neuronios)",
     Pipeline([('sc', StandardScaler()),
               ('nn', MLPClassifier(hidden_layer_sizes=(50,),
                                    activation='relu',
                                    max_iter=500, random_state=42,
                                    early_stopping=True,
                                    validation_fraction=0.1))])),

    ("MLP (2 camadas, 100-50)",
     Pipeline([('sc', StandardScaler()),
               ('nn', MLPClassifier(hidden_layer_sizes=(100, 50),
                                    activation='relu',
                                    max_iter=500, random_state=42,
                                    early_stopping=True,
                                    validation_fraction=0.1))])),

    ("XGBoost (configurado)",
     XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4,
                   reg_lambda=1.0, verbosity=0, random_state=42)),
]

for nome_m, modelo_m in modelos10:
    res_m = cross_validate(modelo_m, X_tr10, y_tr10, cv=kf10, scoring='roc_auc')
    va_m  = res_m['test_score'].mean()
    std_m = res_m['test_score'].std()

    if 'MLP' in nome_m:
        obs = "rede neural -- precisa de mais dado para brilhar"
    elif 'XGBoost' in nome_m:
        obs = "vencedor tipico em dados tabulares"
    else:
        obs = "baseline forte e interpretavel"

    print(f"{nome_m:<35} {va_m:>9.4f} {std_m:>7.4f} | {obs}")

print()
print("CONCLUSAO para dados tabulares:")
print("  XGBoost supera MLP na maioria dos casos com dados estruturados.")
print("  MLP brilha em: imagem, texto, audio, sequencias temporais longas.")
print("  Com dados tabulares e dataset pequeno/medio: XGBoost ou RF primeiro.")
print()
print("QUANDO A REDE NEURAL VENCE EM DADOS TABULARES:")
print("  - Dataset muito grande (> 1M linhas)")
print("  - Features que precisam de embedding (categorias de alta cardinalidade)")
print("  - Interacoes complexas entre muitas features simultaneamente")
print("  - Modelos de linguagem / tabular transformers (ex: TabNet, FT-Transformer)")

In [ ]:
# ============================================================
# QUESTOES DO MODULO 10
# ============================================================

print("=" * 65)
print("QUESTOES DO MODULO 10 -- Resolva antes de ver")
print("=" * 65)
print()
print("QUESTAO 1 (funcoes de ativacao):")
print("  Uma rede neural tem 3 camadas ocultas com ativacao LINEAR em todas.")
print("  (a) O que acontece matematicamente?")
print("  (b) Por que isso e um problema?")
print("  (c) Qual ativacao voce colocaria para resolver?")
print()
print("QUESTAO 2 (escolha de modelo):")
print("  Voce tem um problema de propensao para produto bancario.")
print("  Dataset: 50.000 clientes, 20 features bem estruturadas.")
print("  Seu colega propoe usar deep learning (MLP com 10 camadas).")
print("  O que voce diria?")
print()
print("QUESTAO 3 (identificacao de ativacao):")
print("  Um neuronio tem output de -0.3 com x=-30.")
print("  Quais ativacoes poderiam ter gerado esse output?")
print()
print("-" * 65)
print()
print("GABARITOS:")
print()
print("Q1:")
print("  (a) Matematicamente: Linear(Linear(Linear(x))) = Linear(x).")
print("      Tres camadas lineares equivalem a UMA unica transformacao linear.")
print("      A rede colapsa para y = Wx + b, independente de quantas camadas.")
print()
print("  (b) Problema: perde toda a capacidade de aprender padroes nao-lineares.")
print("      Uma rede neural sem nao-linearidade e apenas uma regressao linear")
print("      disfaracada de complexidade. Nenhuma profundidade adiciona poder.")
print()
print("  (c) ReLU nas camadas ocultas (ou Tanh, Leaky ReLU).")
print("      Sigmoid na saida (para classificacao binaria).")
print("      Softmax na saida (para classificacao multiclasse).")
print()
print("Q2: Eu discordaria da proposta.")
print("  50.000 amostras + 20 features estruturadas = cenario tipico de tabular.")
print("  XGBoost ou Random Forest tipicamente superam MLP nesse contexto.")
print("  Com 10 camadas: alto risco de overfitting com esse volume de dados.")
print("  Alem disso: no Itau, modelos de credito precisam de interpretabilidade")
print("  (BACEN pode exigir explicacao da decisao) -- deep learning e caixa preta.")
print()
print("  Proposta melhor: comecar com logistica (baseline interpretavel),")
print("  depois XGBoost bem configurado. Usar MLP so se ambos ficarem insuficientes.")
print()
print("Q3: Com output = -0.3 e x = -30:")
print("  - ReLU: max(0, -30) = 0, nao -0.3. IMPOSSIVEL.")
print("  - Sigmoid: 1/(1+e^30) ~ 0.000000009, nao -0.3. IMPOSSIVEL.")
print("  - Tanh: tanh(-30) ~ -1.0, nao exatamente -0.3. Possivel em geral.")
print("  - Leaky ReLU: 0.01*(-30) = -0.3. POSSIVEL!")
print("  - Linear: f(-30) = -30, nao -0.3. IMPOSSIVEL com x=-30.")
print()
print("  Conclusao: Leaky ReLU e a ativacao que pode gerar -0.3 com x=-30.")
print("  (Ou qualquer ativacao que aceite valores negativos E produza -0.3 especificamente)")

---
# MODULO 11 -- Clustering e PCA
## Prioridade: ALTA | ISLP Capitulo 12

---

## A diferenca fundamental: supervisionado vs nao-supervisionado

Na Parte 1, todos os modelos precisavam de Y (o rotulo correto).
Clustering e PCA nao precisam de Y -- descobrem estrutura nos proprios dados.

**Analogia:**
Um bibliotecario que nunca viu os livros precisa organiza-los.
Sem saber o "certo" (sem rotulo), ele agrupa por similaridade:
assunto, autor, tamanho, cor da capa...
Isso e clustering: encontrar grupos naturais sem saber a resposta certa.

---

## 11A. K-means Clustering

**Algoritmo:**
1. Inicializar K centroides aleatoriamente
2. Atribuir cada ponto ao centroide mais proximo
3. Recalcular cada centroide como media dos seus pontos
4. Repetir 2-3 ate convergir

**Funcao objetivo (inertia):**
```
inertia = sum_{k=1}^{K} sum_{x in C_k} ||x - centroide_k||^2
```
O K-means minimiza a soma das distancias ao quadrado de cada ponto ao seu centroide.

**[!] CRITICO: SEMPRE normalizar antes do K-means!**
K-means usa distancia euclidiana -- variaveis em maior escala dominam.
Uma variavel em milhoes pode tornar todas as outras irrelevantes.

**Como escolher K:**
- Metodo do cotovelo: plotar inertia vs K, procurar o "joelho"
- Silhouette score: de -1 a 1, maior e melhor (indica clusters bem separados)

---

## 11B. Clustering Hierarquico

Nao precisa definir K antes -- corta o dendrograma onde quiser.
Pode ser bottom-up (aglomerativo) ou top-down (divisivo).

**Linkages -- o que a prova cobra:**

| Linkage | Distancia entre clusters | Sensivel a outliers | Forma |
|---------|--------------------------|--------------------|----|
| **single** | Minima entre pontos | MUITO (cria chains) | Elongado |
| **complete** | Maxima entre pontos | Menos | Compacto |
| **average** | Media par-a-par (NAO e centroide!) | Moderado | Intermediario |
| **ward** | Minimiza variancia intra-cluster | Moderado | Equilibrado |

**[!] Q31 da prova:** average linkage usa media par-a-par, NAO distancia entre centroides!
**[!] Q32 da prova:** complete linkage e MENOS sensivel a outliers que single.
**[!] Q36 da prova:** 'clustering e invariante a normalizacao' -> FALSO!

---

## 11C. PCA (Principal Component Analysis)

**Objetivo:** reduzir dimensionalidade mantendo maxima variancia.

**Como funciona:**
1. Centraliza os dados (media = 0)
2. Calcula a matriz de covariancia
3. Decompoe em autovalores e autovetores
4. Os autovetores sao os componentes principais (direcoes de maxima variancia)
5. Projeta os dados nessas direcoes

**Resultado:** K componentes nao-correlacionados que capturam a maior parte da variancia.

**Quando usar PCA:**
- Visualizacao (reduzir para 2-3 dimensoes)
- Remover multicolinearidade antes de regressao
- Compressao de dados com muitas features

**[!] Sempre normalizar antes do PCA!**
Variaveis em maior escala dominam os componentes.

---

## 11D. Scree Plot -- como escolher o numero de componentes

Plotar a variancia explicada acumulada vs numero de componentes.
Escolher K tal que a variancia acumulada atinja 80-95%.


In [ ]:
# ============================================================
# 11.1 -- K-MEANS: COTOVELO, SILHOUETTE E VISUALIZACAO
# ============================================================

np.random.seed(42)

# Simulando dados de CNPJs com perfis distintos
# Contexto: segmentar a carteira PJ por comportamento financeiro
n11 = 400

# Criando 4 clusters artificiais com caracteristicas diferentes
# Cluster 0: PME digitais (volume baixo, muitas transacoes)
# Cluster 1: Comercio tradicional (volume medio, sazonalidade)
# Cluster 2: Industria (volume alto, poucas transacoes grandes)
# Cluster 3: Servicos (volume medio-baixo, transacoes regulares)

centroides_reais = np.array([
    [2.0, 8.0, 1.0],    # cluster 0: baixo volume, muitas transacoes, pouco credito
    [5.0, 4.0, 5.0],    # cluster 1: medio volume, frequencia media, credito medio
    [9.0, 2.0, 8.0],    # cluster 2: alto volume, poucas transacoes, alto credito
    [3.0, 5.0, 3.0],    # cluster 3: baixo-medio volume, regular, pouco credito
])

dados_cluster = []
for i, centro in enumerate(centroides_reais):
    n_cluster = [120, 100, 80, 100][i]   # tamanhos diferentes
    pts = np.random.normal(centro, 0.8, (n_cluster, 3))
    dados_cluster.append(pts)

X11 = np.vstack(dados_cluster)     # empilhando todos os clusters
labels_reais = np.repeat([0,1,2,3], [120,100,80,100])  # rotulos reais

colunas11 = ['volume_log', 'n_transacoes', 'uso_credito']
df11 = pd.DataFrame(X11, columns=colunas11)

# NORMALIZAR antes do K-means (obrigatorio!)
scaler11 = StandardScaler()
X11_norm = scaler11.fit_transform(X11)

# Testando K de 2 a 8 para escolher o melhor
Ks = range(2, 9)
inertias11 = []    # inertia: menor = mais compacto (mas sempre diminui com mais K)
silhouettes11 = [] # silhouette: maior = melhor separacao (nao precisa diminuir!)

for k in Ks:
    km = KMeans(
        n_clusters=k,
        n_init=15,     # 15 inicializacoes aleatorias (pega a melhor)
        random_state=42
    )
    labels_k = km.fit_predict(X11_norm)   # treina e atribui clusters
    inertias11.append(km.inertia_)         # soma das distancias ao centroide
    silhouettes11.append(silhouette_score(X11_norm, labels_k))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Metodo do Cotovelo
axes[0].plot(Ks, inertias11, 'b-o', ms=8, lw=2)
axes[0].axvline(4, color='red', linestyle='--', lw=2, label='K=4 (cotovelo)')
axes[0].set_xlabel('Numero de clusters K')
axes[0].set_ylabel('Inertia (soma dist. ao centroide)')
axes[0].set_title('Metodo do Cotovelo
Procurar o "joelho" da curva')
axes[0].legend()

# Silhouette Score
axes[1].plot(Ks, silhouettes11, 'g-o', ms=8, lw=2)
axes[1].axvline(4, color='red', linestyle='--', lw=2, label='K=4 (melhor)')
axes[1].set_xlabel('Numero de clusters K')
axes[1].set_ylabel('Silhouette Score (-1 a 1)')
axes[1].set_title('Silhouette Score
Maior = clusters mais separados')
axes[1].legend()

# Visualizacao 2D dos clusters (usando as 2 primeiras variaveis)
km_final = KMeans(n_clusters=4, n_init=15, random_state=42)
labels_final = km_final.fit_predict(X11_norm)

cores11 = ['steelblue', 'coral', 'green', 'purple']
for i in range(4):
    mask = labels_final == i
    axes[2].scatter(X11[mask, 0], X11[mask, 1],
                    color=cores11[i], alpha=0.6, s=30, label=f'Cluster {i}')
# Plotando os centroides (desnormalizados para o espaco original)
centroides_encontrados = scaler11.inverse_transform(km_final.cluster_centers_)
axes[2].scatter(centroides_encontrados[:,0], centroides_encontrados[:,1],
                marker='*', s=300, color='black', label='Centroides')
axes[2].set_xlabel('Volume (log)')
axes[2].set_ylabel('N de transacoes')
axes[2].set_title('K-means (K=4)
SEMPRE normalizar antes!')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

print(f"Melhor K pelo silhouette: {Ks[np.argmax(silhouettes11)]}")
print()
print("[!] PORQUE NORMALIZAR E OBRIGATORIO NO K-MEANS:")
print("  K-means usa distancia euclidiana: dist = sqrt(sum((xi-ci)^2))")
print("  Se 'volume' vai de 0 a 10M e 'n_produtos' vai de 1 a 10,")
print("  a distancia e dominada pelo volume -- n_produtos se torna irrelevante.")
print("  Normalizar (z-score) coloca todas as variaveis na mesma escala.")
print()
print("[!] Q36: 'K-means e invariante a normalizacao' -> FALSO!")
print("  A normalizacao muda completamente os clusters encontrados.")

In [ ]:
# ============================================================
# 11.2 -- CLUSTERING HIERARQUICO E OS 4 LINKAGES
# ============================================================

np.random.seed(42)

# Dataset menor para visualizacao clara do dendrograma
n_hier = 60
X_hier = np.vstack([
    np.random.normal([2,2], 0.4, (n_hier//3, 2)),    # cluster A: canto inferior esquerdo
    np.random.normal([8,8], 0.4, (n_hier//3, 2)),    # cluster B: canto superior direito
    np.random.normal([2,8], 0.4, (n_hier//3, 2)),    # cluster C: canto superior esquerdo
])

# Normalizando antes do hierarchical clustering
X_hier_norm = StandardScaler().fit_transform(X_hier)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

linkages_info = [
    ('single',   'Single Linkage
Distancia MINIMA entre pontos
Sensivel a outliers', axes[0,0]),
    ('complete', 'Complete Linkage
Distancia MAXIMA entre pontos
Menos sensivel (Q32!)', axes[0,1]),
    ('average',  'Average Linkage
Media PAR-A-PAR
NAO e centroide! (Q31!)', axes[1,0]),
    ('ward',     'Ward Linkage
Minimiza variancia intra-cluster
Mais equilibrado', axes[1,1]),
]

for method, titulo, ax in linkages_info:
    Z = linkage(X_hier_norm, method=method)

    # Dendrograma: arvore que mostra como os pontos sao agrupados
    # Eixo Y: distancia na qual dois clusters sao fundidos
    # Linha horizontal: fusao de dois clusters
    dendrogram(
        Z,
        ax=ax,
        no_labels=True,                    # sem labels nos pontos (muito cheio)
        color_threshold=0.7*max(Z[:,2])    # colore clusters acima do threshold
    )
    ax.set_title(titulo, fontsize=11)
    ax.set_ylabel('Distancia de fusao')

plt.suptitle('Os 4 Linkages do Hierarchical Clustering', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print("RESUMO DOS LINKAGES -- O QUE A PROVA COBRA:")
print()

linkage_summary = [
    ("single",   "Minima distancia entre pontos", "MUITO sensivel",  "Chains elongados"),
    ("complete", "Maxima distancia entre pontos", "Menos sensivel",  "Clusters compactos"),
    ("average",  "Media par-a-par (NAO centroide!)", "Moderado",     "Intermediario"),
    ("ward",     "Minimiza variancia intra-cluster", "Moderado",     "Clusters equilibrados"),
]

print(f"{'Linkage':<12} {'Distancia':<35} {'Outlier':>15} {'Resultado':<25}")
print("-" * 92)
for lk, dist, out, res in linkage_summary:
    print(f"{lk:<12} {dist:<35} {out:>15} {res:<25}")

print()
print("[!] Q31: average linkage usa MEDIA PAR-A-PAR, nao centroide.")
print("  Para distancia entre centroides: use 'centroid' linkage (menos comum).")
print()
print("[!] Q32: complete linkage e MENOS sensivel a outliers que single.")
print("  Single: um outlier pode criar uma 'corrente' longa (chaining effect).")
print("  Complete: usa a distancia maxima -- um outlier nao domina o resultado.")

In [ ]:
# ============================================================
# 11.3 -- Q30 DA PROVA: SINGLE LINKAGE PARA K GRUPOS
# ============================================================

if os.path.exists('agrupamento.csv'):
    df_q30 = pd.read_csv('agrupamento.csv')
    Z_q30  = linkage(df_q30.values, method='single')

    print("=== Q30 DA PROVA 2019: Single Linkage, limiar para 4 grupos ===")
    print()
    print("Distancias de fusao e numero de grupos resultantes:")
    print()
    for d in sorted(set(Z_q30[:,2])):
        ng = len(set(fcluster(Z_q30, d, criterion='distance')))
        if 3 <= ng <= 6:
            print(f"  Limiar = {d:.4f} -> {ng} grupos")

    print()
    print("Para 4 grupos: limiar entre 2.21 e 4.63")
    print("Candidato marcou '0.5 < d < 2.2' -> ERRADO (esse limiar gera 5 grupos, nao 4!)")
    print("Resposta correta: faixa que começa acima de 2.21")
else:
    print("Coloque agrupamento.csv na mesma pasta para reproduzir Q30.")
    print()
    print("LOGICA DA Q30:")
    print("  O dendrograma mostra distancias nas quais clusters sao fundidos.")
    print("  Cortar o dendrograma em um limiar d:")
    print("    - Pequeno: muitos grupos (cada ponto e um cluster)")
    print("    - Grande:  poucos grupos (todos no mesmo cluster)")
    print("  Para K grupos: encontrar o limiar onde exatamente K clusters existem.")
    print("  Verificar: fcluster(Z, limiar, criterion='distance') deve dar K grupos.")

In [ ]:
# ============================================================
# 11.4 -- PCA: SCREE PLOT E VISUALIZACAO
# ============================================================

np.random.seed(42)

# Simulando dados de clientes PJ com 10 variaveis correlacionadas
# Mas com apenas 3 dimensoes latentes reais (3 fatores ocultos)
n_pca = 300
grupos_pca = np.random.choice([0, 1, 2], n_pca, p=[0.4, 0.3, 0.3])

# Centroides nos fatores latentes
centros_pca = np.array([
    [3, 3, 0, 0, 0, 0, 0, 0, 0, 0],    # grupo 0
    [-3, -3, 0, 0, 0, 0, 0, 0, 0, 0],  # grupo 1
    [0, 0, 3, -3, 0, 0, 0, 0, 0, 0],   # grupo 2
])
X_pca = np.array([centros_pca[g] + np.random.normal(0, 1, 10) for g in grupos_pca])

# NORMALIZAR antes do PCA (obrigatorio!)
scaler_pca = StandardScaler()
X_pca_norm = scaler_pca.fit_transform(X_pca)

# Aplicando PCA
pca = PCA()           # sem especificar n_components: calcula todos
pca.fit(X_pca_norm)   # calcula os componentes no dado normalizado

# Projecao nos 2 primeiros componentes (para visualizacao)
X_pca_2d = pca.transform(X_pca_norm)[:, :2]

# Variancia explicada por cada componente
var_exp = pca.explained_variance_ratio_     # proporcao da variancia de cada PC
var_cum = np.cumsum(var_exp)               # variancia acumulada

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Scree Plot: variancia explicada por componente
axes[0].bar(range(1, len(var_exp)+1), var_exp*100, color='steelblue', label='Individual')
axes[0].plot(range(1, len(var_exp)+1), var_cum*100, 'ro-', ms=5, label='Acumulada')
axes[0].axhline(80, color='green', linestyle='--', lw=1.5, label='80% variancia')
axes[0].axhline(95, color='orange', linestyle='--', lw=1.5, label='95% variancia')
axes[0].set_xlabel('Componente principal')
axes[0].set_ylabel('Variancia explicada (%)')
axes[0].set_title('Scree Plot
Quantos componentes usar?')
axes[0].legend(fontsize=8)

# Projecao 2D colorida pelos grupos reais
cores_pca = ['blue', 'red', 'green']
for g in [0, 1, 2]:
    mask_g = grupos_pca == g
    axes[1].scatter(X_pca_2d[mask_g, 0], X_pca_2d[mask_g, 1],
                    color=cores_pca[g], alpha=0.6, s=30, label=f'Grupo {g}')
axes[1].set_xlabel(f'PC1 ({var_exp[0]*100:.1f}% da variancia)')
axes[1].set_ylabel(f'PC2 ({var_exp[1]*100:.1f}% da variancia)')
axes[1].set_title('Projecao 2D via PCA
Estrutura oculta revelada')
axes[1].legend()

# Loadings do PC1: contribuicao de cada variavel original
loadings_pc1 = pd.Series(pca.components_[0],
                          index=[f'X{i}' for i in range(10)])
loadings_pc1.abs().sort_values(ascending=True).plot(
    kind='barh', ax=axes[2], color='coral'
)
axes[2].set_title('Loadings do PC1
Quais variaveis contribuem mais?')
axes[2].set_xlabel('|Loading|')

plt.tight_layout()
plt.show()

# Quantos componentes para 80% e 95%?
n_80 = np.searchsorted(var_cum, 0.80) + 1
n_95 = np.searchsorted(var_cum, 0.95) + 1

print(f"=== VARIANCIA EXPLICADA POR COMPONENTE ===")
print()
for i, (ind, cum) in enumerate(zip(var_exp[:6], var_cum[:6])):
    print(f"  PC{i+1}: {ind*100:.1f}% individual | {cum*100:.1f}% acumulada")

print(f"  ...")
print()
print(f"Para >= 80% da variancia: {n_80} componentes de 10")
print(f"Para >= 95% da variancia: {n_95} componentes de 10")
print()
print("[!] COMO ESCOLHER O NUMERO DE COMPONENTES:")
print("  1. Criterio da variancia: manter >= 80% (ou 95%) da variancia")
print("  2. Criterio do cotovelo: onde a curva do scree plot 'dobra'")
print("  3. Criterio do autovalor > 1 (Kaiser): manter componentes com variancia > media")
print()
print("[!] SEMPRE normalizar antes do PCA!")
print("  Variaveis em maior escala dominam os componentes.")
print("  Ex: 'volume_transac' em milhoes dominaria todos os PCs sem normalizar.")

In [ ]:
# ============================================================
# QUESTOES DO MODULO 11
# ============================================================

print("=" * 65)
print("QUESTOES DO MODULO 11 -- Resolva antes de ver")
print("=" * 65)
print()
print("QUESTAO 1 (linkages -- pegadinha classica):")
print("  Assinale VERDADEIRO ou FALSO:")
print("  (a) Average linkage usa a distancia entre os centroides dos clusters")
print("  (b) Complete linkage e mais sensivel a outliers que single linkage")
print("  (c) K-means e invariante a normalizacao dos dados")
print("  (d) Clustering e um algoritmo supervisionado")
print("  (e) Ward linkage minimiza a variancia intra-cluster")
print()
print("QUESTAO 2 (escolha do numero de clusters):")
print("  Scree plot do silhouette score mostra:")
print("  K=2: 0.45 | K=3: 0.61 | K=4: 0.58 | K=5: 0.52 | K=6: 0.49")
print("  E o metodo do cotovelo mostra joelho em K=4.")
print("  Qual K voce escolheria e por que?")
print()
print("QUESTAO 3 (PCA):")
print("  PCA em 20 variaveis: os 4 primeiros PCs explicam 82% da variancia.")
print("  (a) Voce usaria os 20 originais ou os 4 PCs como input do modelo?")
print("  (b) Quais as vantagens e desvantagens de usar os 4 PCs?")
print()
print("-" * 65)
print()
print("GABARITOS:")
print()
print("Q1:")
print("  (a) FALSO. Average usa MEDIA PAR-A-PAR entre todos os pares de pontos.")
print("      Distancia entre centroides seria 'centroid linkage' (diferente!).")
print()
print("  (b) FALSO. Single linkage e MAIS sensivel a outliers que complete.")
print("      Complete usa distancia MAXIMA -- um outlier nao cria 'pontes' longas.")
print("      Single usa distancia MINIMA -- um outlier pode criar chains artificiais.")
print()
print("  (c) FALSO. K-means usa distancia euclidiana.")
print("      Sem normalizar, variaveis em maior escala dominam o resultado.")
print("      A normalizacao muda completamente os clusters encontrados.")
print()
print("  (d) FALSO. Clustering e NAO-SUPERVISIONADO.")
print("      Nao usa rotulos. Descobre estrutura nos proprios dados.")
print("      Se usasse rotulos, seria CLASSIFICACAO (supervisionado).")
print()
print("  (e) VERDADEIRO. Ward linkage minimiza a variancia intra-cluster")
print("      em cada fusao, gerando clusters mais compactos e equilibrados.")
print()
print("Q2: K=3.")
print("  Silhouette: maximo em K=3 (0.61). Maior silhouette = melhor separacao.")
print("  Cotovelo: joelho em K=4, mas o silhouette diz K=3.")
print("  Conflito: priorizar silhouette (metrica mais objetiva).")
print("  Contudo: verificar com o negocio se K=3 ou K=4 faz mais sentido")
print("  para segmentacao da carteira (interpretabilidade importa!).")
print()
print("Q3:")
print("  (a) Depende do objetivo.")
print("      Se o objetivo e reducao de dimensionalidade e os 4 PCs capturam")
print("      82% da variancia, use os 4 PCs -- reduz ruido e multicolinearidade.")
print("      Se interpretabilidade e critica (BACEN), use as 20 originais")
print("      com regularizacao (Lasso para selecao).")
print()
print("  (b) Vantagens dos 4 PCs:")
print("      - Sem multicolinearidade (componentes sao ortogonais entre si)")
print("      - Reducao de ruido (captura apenas as direcoes de maior variancia)")
print("      - Menos features = modelo mais rapido e menos propenso a overfit")
print()
print("      Desvantagens dos 4 PCs:")
print("      - Perda de 18% da variancia")
print("      - Componentes nao sao interpretaveis (combinacao linear de variaveis)")
print("      - Nao da para explicar para o negocio: 'PC1 indica que...'")
print("      - Em modelos regulados: o gestor precisa entender cada input")

---
# MODULO 12 -- MLflow: Ciclo de Vida do Modelo
## Prioridade: MEDIA (fundamental para entrevista e producao)

---

## A analogia do laboratorio de pesquisa

Imagine um laboratorio sem caderno de experimentos.
Cada cientista treina um modelo, testa, esquece os parametros.
Na reuniao, ninguem sabe qual foi o melhor resultado ou como reproduzir.
O modelo que foi para producao: qual versao era? Com quais dados?

MLflow e o caderno de laboratorio digital do cientista de dados.
Registra tudo automaticamente: parametros, metricas, dados, modelo.

---

## 12A. Os 4 componentes

**Tracking:** registra cada experimento.
- `mlflow.log_param('learning_rate', 0.05)`
- `mlflow.log_metric('auc_validacao', 0.83)`
- `mlflow.sklearn.log_model(modelo, 'modelo_propensao')`

**Models:** padroniza como modelos sao salvos e carregados.
Funciona com sklearn, XGBoost, PyTorch, TensorFlow.

**Model Registry:** versiona e gerencia modelos em producao.
- None -> Staging (testando) -> Production (ativo) -> Archived
- Permite rollback imediato se o novo modelo for pior.

**Projects:** empacota codigo + dependencias + parametros.
Garante reproducibilidade: qualquer um consegue rodar o experimento.

---

## 12B. O que o Itau vai perguntar na entrevista

- "Como voce gerencia versoes de modelo em producao?"
  -> MLflow Tracking + Model Registry
  -> Cada modelo tem: versao, data, AUC, parametros, dados usados

- "Como voce sabe se o modelo degradou?"
  -> Monitoring de drift (PSI, KS test) -- nao e MLflow nativo
  -> MLflow rastreia a performance mas nao monitora distribuicoes de input

- "Como voce garante reproducibilidade?"
  -> MLflow Projects + log dos dados de treino (hash ou versao)

---

## 12C. PSI -- Population Stability Index

Mede o quanto a distribuicao de uma variavel mudou entre treino e producao.

```
PSI = sum[ (real - esperado) * ln(real/esperado) ]
```

- PSI < 0.10: mudanca insignificante -- modelo ainda valido
- PSI 0.10-0.25: mudanca moderada -- investigar
- PSI > 0.25: mudanca severa -- **retreinar o modelo!**

Contexto bancario: após uma crise economica, o PSI do 'volume_mensal'
pode disparar para 0.4+ -- o modelo de propensao foi treinado numa
distribuicao completamente diferente da atual.


In [ ]:
# ============================================================
# 12.1 -- MLFLOW TRACKING COMPLETO
# ============================================================

np.random.seed(42)

# Dataset de propensao para produto bancario
X12, y12 = make_classification(
    n_samples=1200, n_features=15, n_informative=8, random_state=42
)
X_tr12, X_te12, y_tr12, y_te12 = train_test_split(
    X12, y12, test_size=0.2, random_state=42
)
kf12 = KFold(5, shuffle=False)

# Configurando o experimento no MLflow
mlflow.set_experiment('propensao_pj_v1')

modelos12 = [
    {
        'nome': 'LogReg_baseline',
        'modelo': Pipeline([('sc', StandardScaler()),
                            ('lr', LogisticRegression(C=1.0, max_iter=1000))]),
        'params': {'tipo': 'LogisticRegression', 'C': 1.0, 'scaled': True}
    },
    {
        'nome': 'RF_depth8',
        'modelo': RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42),
        'params': {'tipo': 'RandomForest', 'n_trees': 200, 'max_depth': 8}
    },
    {
        'nome': 'XGB_lr005',
        'modelo': XGBClassifier(n_estimators=500, learning_rate=0.05,
                                max_depth=4, reg_lambda=1.0,
                                verbosity=0, random_state=42),
        'params': {'tipo': 'XGBoost', 'lr': 0.05, 'depth': 4, 'lambda': 1.0}
    },
]

resultados12 = []

for cfg in modelos12:
    # Cada 'run' e uma execucao de experimento no MLflow
    with mlflow.start_run(run_name=cfg['nome']):

        # 1. Registrar parametros ANTES de treinar
        mlflow.log_params(cfg['params'])

        # 2. CV para estimativa honesta
        res12 = cross_validate(
            cfg['modelo'], X_tr12, y_tr12,
            cv=kf12, scoring='roc_auc',
            return_train_score=True
        )
        auc_tr12 = res12['train_score'].mean()
        auc_va12 = res12['test_score'].mean()
        auc_sd12 = res12['test_score'].std()

        # 3. Treinar no conjunto de treino completo
        cfg['modelo'].fit(X_tr12, y_tr12)

        # 4. Avaliar no conjunto de teste (uma unica vez!)
        probas12 = cfg['modelo'].predict_proba(X_te12)[:, 1]
        auc_te12 = roc_auc_score(y_te12, probas12)
        ll12     = log_loss(y_te12, probas12)

        # 5. Registrar metricas
        mlflow.log_metric('auc_treino_cv',  round(auc_tr12, 4))
        mlflow.log_metric('auc_val_cv',     round(auc_va12, 4))
        mlflow.log_metric('auc_val_std',    round(auc_sd12, 4))
        mlflow.log_metric('auc_teste',      round(auc_te12, 4))
        mlflow.log_metric('log_loss_teste', round(ll12, 4))
        mlflow.log_metric('gap_overfit',    round(auc_tr12 - auc_va12, 4))

        # 6. Salvar o modelo (serializa e associa ao run)
        mlflow.sklearn.log_model(cfg['modelo'], 'modelo')

        resultados12.append({
            'Modelo':    cfg['nome'],
            'AUC Tr CV': round(auc_tr12, 4),
            'AUC Val CV': f"{auc_va12:.4f} +/- {auc_sd12:.4f}",
            'AUC Teste': round(auc_te12, 4),
            'Log Loss':  round(ll12, 4),
            'Gap':       round(auc_tr12 - auc_va12, 4)
        })

print("=== RESULTADOS COMPARATIVOS ===")
print()
df_res = pd.DataFrame(resultados12)
print(df_res.to_string(index=False))

print()
print("Para visualizar no MLflow UI:")
print("  Execute no terminal: mlflow ui")
print("  Acesse: http://localhost:5000")
print()
print("BOAS PRATICAS DE MLFLOW:")
print("  1. Sempre log_params ANTES de treinar (rastreabilidade)")
print("  2. Sempre logar AUC de TREINO E VALIDACAO (para ver gap de overfit)")
print("  3. Salvar o modelo com log_model (reproducibilidade)")
print("  4. Usar nomes de run descritivos: 'XGB_lr005_depth4_lambda1'")
print("  5. Adicionar tags: mlflow.set_tag('autor', 'lucas')")
print("  6. Logar o hash dos dados de treino: mlflow.log_param('data_hash', hash_treino)")

In [ ]:
# ============================================================
# 12.2 -- PSI: MONITORAMENTO DE DRIFT EM PRODUCAO
# ============================================================

print("=== PSI: POPULATION STABILITY INDEX ===")
print()
print("Mede o quanto a distribuicao de uma variavel mudou entre treino e producao.")
print()

def calcular_psi(ref, atual, n_bins=10):
    # PSI = sum[ (P_atual - P_ref) * ln(P_atual / P_ref) ]
    # onde P_ref e a proporcao em cada bin no dataset de referencia (treino)
    # e P_atual e a proporcao em cada bin no dataset atual (producao)

    # Criando bins baseados nos percentis do dado de referencia
    bins = np.percentile(ref, np.linspace(0, 100, n_bins + 1))
    bins[0]  = -np.inf   # extremo inferior aberto
    bins[-1] = +np.inf   # extremo superior aberto

    # Calculando frequencias em cada bin
    freq_ref   = np.histogram(ref,   bins=bins)[0] / len(ref)
    freq_atual = np.histogram(atual, bins=bins)[0] / len(atual)

    # Evitando log(0) com valor pequeno
    freq_ref   = np.where(freq_ref   == 0, 0.0001, freq_ref)
    freq_atual = np.where(freq_atual == 0, 0.0001, freq_atual)

    psi = np.sum((freq_atual - freq_ref) * np.log(freq_atual / freq_ref))
    return psi

np.random.seed(42)
n_base = 2000

# Simulando features do modelo de propensao em producao
features_sim = {
    'volume_mensal':    (np.random.lognormal(7, 1.5, n_base),
                         np.random.lognormal(8.0, 1.5, n_base)),  # drift: media subiu
    'n_produtos':       (np.random.poisson(3, n_base),
                         np.random.poisson(3.2, n_base)),           # drift pequeno
    'score_pj':         (np.random.normal(650, 80, n_base),
                         np.random.normal(620, 90, n_base)),        # drift moderado
    'sazonalidade':     (np.random.binomial(1, 0.4, n_base),
                         np.random.binomial(1, 0.45, n_base)),      # drift minimo
    'inadimplencia_90': (np.random.binomial(1, 0.08, n_base),
                         np.random.binomial(1, 0.18, n_base)),      # drift SEVERO (crise!)
}

print(f"{'Feature':<20} {'PSI':>8} {'Status':<15} {'Acao'}")
print("-" * 68)

for feat, (ref, atual) in features_sim.items():
    psi = calcular_psi(ref, atual)

    if psi < 0.10:
        status = "OK"
        acao   = "Nenhuma"
    elif psi < 0.25:
        status = "ATENCAO"
        acao   = "Investigar"
    else:
        status = "RETREINAR!"
        acao   = "URGENTE: retreinar o modelo"

    print(f"{feat:<20} {psi:>8.4f} {status:<15} {acao}")

print()
print("INTERPRETACAO DO PSI:")
print("  < 0.10: mudanca insignificante -- modelo ainda valido")
print("  0.10-0.25: mudanca moderada -- investigar causa, monitorar mais frequente")
print("  > 0.25: mudanca severa -- retreinar com dados recentes!")
print()
print("NO CONTEXTO BANCARIO:")
print("  'inadimplencia_90' com PSI=0.4+ indica crise economica ou mudanca de perfil.")
print("  O modelo foi treinado com taxa de inadimplencia de 8%, agora e 18%.")
print("  A relacao entre features e target mudou -- modelo e invalido.")
print()
print("QUANDO RETREINAR:")
print("  1. PSI de qualquer feature > 0.25")
print("  2. AUC de validacao caiu mais de 5pp em relacao ao baseline")
print("  3. Calibracao deteriorou (taxas reais vs previstas divergindo)")
print("  4. Mudanca regulatoria ou de politica de credito")

In [ ]:
# ============================================================
# QUESTOES DO MODULO 12
# ============================================================

print("=" * 65)
print("QUESTOES DO MODULO 12 -- Resolva antes de ver")
print("=" * 65)
print()
print("QUESTAO 1 (boas praticas):")
print("  Seu colega registrou um experimento no MLflow assim:")
print("  - Treinou o modelo")
print("  - Olhou a AUC de validacao: 0.82")
print("  - Ajustou os parametros")
print("  - Re-treinou")
print("  - AUC subiu para 0.86")
print("  - Registrou no MLflow apenas o segundo experimento (AUC=0.86)")
print()
print("  O que esta errado? Como fazer corretamente?")
print()
print("QUESTAO 2 (PSI e tomada de decisao):")
print("  Monitoramento mensal do modelo de churn detectou:")
print("  PSI de 'volume_mensal' = 0.31")
print("  AUC de validacao caiu de 0.83 para 0.79 (queda de 4pp)")
print()
print("  (a) O que cada indicador sugere?")
print("  (b) Voce retreinaria imediatamente? Quais perguntas faria primeiro?")
print()
print("QUESTAO 3 (ciclo de vida):")
print("  O que e o Model Registry do MLflow e por que ele importa")
print("  em um ambiente de producao como o do Itau?")
print()
print("-" * 65)
print()
print("GABARITOS:")
print()
print("Q1: Varios problemas:")
print("  1. Registrou apenas o 'vencedor' -- perdeu a rastreabilidade.")
print("     Se precisar investigar por que o modelo 1 era pior, nao tem registro.")
print("  2. Ajustar parametros DEPOIS de ver a AUC de validacao e data snooping.")
print("     A AUC de validacao ja foi contaminada pelo ajuste.")
print("     Correto: separar um conjunto de teste final e so usar uma vez.")
print()
print("  COMO FAZER CORRETAMENTE:")
print("  - Registrar TODOS os experimentos (com mlflow.start_run() em cada tentativa)")
print("  - Usar CV para escolher parametros (nao o conjunto de teste)")
print("  - Avaliar no conjunto de teste APENAS para reportar performance final")
print("  - Tags para identificar: 'experimento_1', 'experimento_2', etc.")
print()
print("Q2:")
print("  (a) PSI=0.31: mudanca SEVERA na distribuicao do volume mensal.")
print("      Os dados de producao sao muito diferentes do treino.")
print("      AUC caindo 4pp: degradacao real de performance, mas ainda aceitavel.")
print()
print("  (b) Nao necessariamente imediato. Perguntas antes de retreinar:")
print("      - A queda de 4pp impacta o negocio? (custo da decisao errada)")
print("      - A mudanca no volume e temporaria (sazonalidade) ou permanente?")
print("      - Quais outros features estao com PSI alto?")
print("      - A calibracao do modelo esta aceitavel?")
print("      - Ha dados suficientes recentes para retreinar com qualidade?")
print("      Se a queda for em janela sazonal (ex: janeiro): monitorar.")
print("      Se for tendencia estrutural: retreinar urgente.")
print()
print("Q3: Model Registry e o componente do MLflow que:")
print("  - Versiona cada modelo salvo (v1, v2, v3...)")
print("  - Gerencia o ciclo de vida: None -> Staging -> Production -> Archived")
print("  - Permite rollback em 1 comando se o novo modelo for pior")
print("  - Registra quem aprovou a promocao para producao (auditoria)")
print()
print("  No Itau (ambiente regulado pelo BACEN):")
print("  - Auditabilidade: saber exatamente qual modelo tomou cada decisao")
print("  - Rastreabilidade: quais dados foram usados, em que data, por quem")
print("  - Rollback rapido: se o modelo XGBoost_v3 piorar, voltar para v2 em minutos")
print("  - Aprovacao formal: gestor aprova promocao de Staging para Production")